# Explainable AI via Shapley Values — From Scratch
## for a Simulated Credit Score Neural Network

_Generated: 2025-10-22T18:01:33.568562Z_

We build a small **neural network** (from scratch with NumPy) to predict **credit approval risk** from simulated tabular features. We then compute **feature attributions** for individual decisions using **Shapley values** via **permutation sampling** (unbiased Monte‑Carlo estimator). We avoid external explainability toolkits and implement the logic directly.

**Highlights**
- Synthetic credit-like features; train/test split and standardization
- 2‑layer MLP (ReLU→Sigmoid) trained with SGD
- **Shapley** attributions for a chosen instance using a **background baseline** and random permutations
- Global importances (mean absolute Shapley), dependence and waterfall plots
- Saved artifacts + one‑click download cell

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports & Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

def train_test_split(X, y, test_size=0.25, rng=rng):
    n = len(y); idx = np.arange(n); rng.shuffle(idx)
    n_te = int(np.round(test_size*n))
    te = idx[:n_te]; tr = idx[n_te:]
    return X[tr], X[te], y[tr], y[te]

def standardize(X, mean=None, std=None, eps=1e-12):
    if mean is None: mean = X.mean(axis=0)
    if std is None: std = X.std(axis=0)
    std = np.where(std < eps, 1.0, std)
    return (X-mean)/std, mean, std

## 2) Simulated Credit Data

We simulate a tabular dataset with features: income_k, dti, util, hist_len, inq_12m, delinq. A latent score produces approval probability via a sigmoid.

In [ ]:
def simulate_credit(n=4000, rng=rng):
    income = rng.lognormal(mean=3.5, sigma=0.4, size=n)
    dti = np.clip(rng.normal(0.28, 0.1, size=n), 0.01, 0.8)
    util = np.clip(rng.beta(2,5, size=n), 0.0, 0.99)
    hist = np.clip(rng.normal(6.0, 3.0, size=n), 0.0, 30.0)
    inq  = np.clip(rng.poisson(2.0, size=n), 0, 10)
    delq = np.clip((rng.poisson(0.6, size=n) > 0).astype(float) * rng.poisson(1.2, size=n), 0, 8)

    X = np.c_[income, dti, util, hist, inq, delq].astype(float)
    z = (
        + 0.003 * income
        - 2.2   * dti
        - 1.6   * util
        + 0.12  * hist
        - 0.20  * inq
        - 0.35  * delq
        + 0.8   * np.maximum(0, 0.35 - util)
        - 0.6   * np.maximum(0, dti - 0.45)
    )
    z += 0.4*np.tanh((income/50.0 - 1.0))
    p = 1/(1+np.exp(-z))
    y = (rng.random(n) < p).astype(float)
    return X, y, ["income_k", "dti", "util", "hist_len", "inq_12m", "delinq"]

X, y, feature_names = simulate_credit(5000, rng)
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X, y, test_size=0.25, rng=rng)
X_tr, mu, sig = standardize(X_tr_raw)
X_te, _, _ = standardize(X_te_raw, mu, sig)
d = X_tr.shape[1]
print("Train:", X_tr.shape, "| Test:", X_te.shape, "| Pos rate (train):", y_tr.mean())

## 3) Neural Network (NumPy)

In [ ]:
class MLP:
    def __init__(self, d_in, h=16, seed=SEED):
        rng_local = np.random.default_rng(seed)
        self.W1 = rng_local.normal(0, np.sqrt(2.0/d_in), size=(d_in, h))
        self.b1 = np.zeros(h)
        self.W2 = rng_local.normal(0, np.sqrt(2.0/h), size=(h, 1))
        self.b2 = np.zeros(1)

    @staticmethod
    def relu(x): return np.maximum(0, x)
    @staticmethod
    def sigmoid(z): return 1.0/(1.0 + np.exp(-np.clip(z, -40, 40)))

    def forward(self, X):
        H = self.relu(X @ self.W1 + self.b1)
        Z = H @ self.W2 + self.b2
        P = self.sigmoid(Z)
        cache = {"X": X, "H": H, "Z": Z, "P": P}
        return P, cache

    def loss_and_grads(self, X, y):
        P, cache = self.forward(X)
        eps = 1e-12
        y = y.reshape(-1,1)
        L = -np.mean(y*np.log(P+eps) + (1-y)*np.log(1-P+eps))
        dZ = (P - y) / len(y)
        dW2 = cache["H"].T @ dZ
        db2 = np.sum(dZ, axis=0)
        dH = dZ @ self.W2.T
        dH[cache["H"]<=0] = 0.0
        dW1 = cache["X"].T @ dH
        db1 = np.sum(dH, axis=0)
        grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
        return L, grads

    def step(self, grads, lr=1e-2):
        self.W1 -= lr*grads["dW1"]
        self.b1 -= lr*grads["db1"]
        self.W2 -= lr*grads["dW2"]
        self.b2 -= lr*grads["db2"]

    def predict_proba(self, X):
        return self.forward(X)[0].ravel()

mlp = MLP(d_in=d, h=16, seed=SEED)

## 4) Training Loop & Evaluation

In [ ]:
def roc_auc(y_true, scores):
    # trapezoid on ROC constructed from thresholds
    o = np.argsort(scores)
    P = np.sum(y_true==1); N = np.sum(y_true==0)
    if P==0 or N==0: return np.nan
    t = np.r_[np.inf, np.unique(scores)[::-1], -np.inf]
    tpr=[]; fpr=[]
    for thr in t:
        m = scores>=thr
        tp = np.sum((y_true==1)&m); fp = np.sum((y_true==0)&m)
        tpr.append(tp/(P+1e-12)); fpr.append(fp/(N+1e-12))
    tpr=np.array(tpr); fpr=np.array(fpr)
    order=np.argsort(fpr)
    return np.trapz(tpr[order], fpr[order])

def train(model, X, y, Xv, yv, lr=7e-3, epochs=80, batch=256):
    n = len(y)
    hist = {"loss": [], "vauc": []}
    for ep in range(1, epochs+1):
        idx = np.arange(n); rng.shuffle(idx)
        for i0 in range(0, n, batch):
            j = idx[i0:i0+batch]
            L, g = model.loss_and_grads(X[j], y[j])
            model.step(g, lr=lr)
        p = model.predict_proba(Xv)
        auc = roc_auc(yv, p)
        hist["loss"].append(L); hist["vauc"].append(auc)
        if ep%10==0:
            print(f"epoch {ep:02d}  loss={L:.4f}  val AUC={auc:.3f}")
    return hist

hist = train(mlp, X_tr, y_tr, X_te, y_te, lr=7e-3, epochs=80, batch=256)
p_te = mlp.predict_proba(X_te)
yhat = (p_te>=0.5).astype(float)
acc = (yhat==y_te).mean()
print(f"Test accuracy: {acc:.3f}")

### Learning Curves

In [ ]:
fig = plt.figure(figsize=(7,4))
plt.plot(hist["loss"], label="train BCE")
plt.plot(hist["vauc"], label="val AUC")
plt.xlabel("Epoch"); plt.title("Training progress"); plt.legend()
plt.tight_layout(); plt.show()

## 5) Shapley Values via Permutation Sampling (from scratch)

In [ ]:
def shapley_permutation(f_predict, x, background, M=256, rng=rng):
    d = x.shape[0]
    base_value = float(f_predict(background[None,:])[0])
    fx = float(f_predict(x[None,:])[0])
    phi = np.zeros(d)
    for m in range(M):
        perm = rng.permutation(d)
        cur = background.copy()
        prev_out = base_value
        for j in perm:
            cur[j] = x[j]
            out = float(f_predict(cur[None,:])[0])
            phi[j] += (out - prev_out)
            prev_out = out
    phi /= M
    return phi, base_value, fx

# Background: mean over a random subset
K_BG = 200
bg_idx = rng.choice(len(X_tr), size=min(K_BG, len(X_tr)), replace=False)
background = X_tr[bg_idx].mean(axis=0)

i_explain = 5
x0 = X_te[i_explain]
phi, f0, fx = shapley_permutation(mlp.predict_proba, x0, background, M=400, rng=rng)
print("Base f(z):", f0, "| f(x):", fx, "| sum(phi):", phi.sum(), "=> f(z)+sum(phi)≈", f0+phi.sum())

## 6) Visualizing a Local Explanation

In [ ]:
names = np.array(feature_names)
x0_raw = x0*sig + mu
order = np.argsort(np.abs(phi))[::-1]
names_ord = names[order]; phi_ord = phi[order]

fig = plt.figure(figsize=(7,4))
plt.bar(range(len(phi_ord)), phi_ord)
plt.xticks(range(len(phi_ord)), names_ord, rotation=30)
plt.ylabel("Contribution to output probability")
plt.title("Local Shapley Contributions (x₀)")
plt.tight_layout(); plt.show()

print("Prediction f(x0) =", fx, "  Base =", f0)
for n, v in zip(names_ord, phi_ord):
    print(f"{n:>10s}: {v:+.4f}")

## 7) Global Importance via Mean |Shapley|

In [ ]:
def batch_shapley_summary(model, X, M=120, bg=None, rng=rng):
    if bg is None: bg = X.mean(axis=0)
    Phi = np.zeros((len(X), X.shape[1]))
    for i in range(len(X)):
        ph, f0i, fxi = shapley_permutation(model.predict_proba, X[i], bg, M=M, rng=rng)
        Phi[i] = ph
    return Phi

subset = rng.choice(len(X_te), size=60, replace=False)
Phi = batch_shapley_summary(mlp, X_te[subset], M=160, bg=background, rng=rng)
imp = np.mean(np.abs(Phi), axis=0)

fig = plt.figure(figsize=(6,4))
ordg = np.argsort(imp)[::-1]
plt.bar(range(len(imp)), imp[ordg])
plt.xticks(range(len(imp)), names[ordg], rotation=30)
plt.ylabel("Mean |Shapley|"); plt.title("Global feature importance (approx)")
plt.tight_layout(); plt.show()

## 8) Dependence Plots

In [ ]:
topk = 3
for k in range(topk):
    j = ordg[k]
    xs = X_te[subset, j]*sig[j] + mu[j]
    ys = Phi[:, j]
    fig = plt.figure(figsize=(5,3.5))
    plt.scatter(xs, ys, s=14)
    plt.xlabel(feature_names[j] + " (raw units)")
    plt.ylabel("Shapley φ_j")
    plt.title(f"Dependence: {feature_names[j]}")
    plt.tight_layout(); plt.show()

## 9) Save Artifacts & Download

In [ ]:
import os, json as _json
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/credit_shap_from_scratch.npz",
         feature_names=np.array(feature_names, dtype=object),
         mu=mu, sig=sig,
         model_W1=mlp.W1, model_b1=mlp.b1,
         model_W2=mlp.W2, model_b2=mlp.b2,
         background=background,
         x0=x0, phi=phi, base_value=np.array([f0]), fx=np.array([fx]),
         global_importance=imp, Phi=Phi, subset_idx=subset)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 10) Notes & Extensions

- Use multiple **background rows** (e.g., k-medoids) and average per‑row Shapley to reduce imputation bias.
- Increase `M` for lower variance; consider **KernelSHAP** style weighting via local linear regression.
- Explain **log‑odds** instead of probability for better additivity on saturated outputs.
- Add fairness probes: perturb or mask potentially sensitive features in the background to test stability.